# Linear & Quadratic Discriminant Analysis (LDA / QDA)

A complete guide: intuition, mathematics, and implementation from scratch.

## Table of Contents

1. [Intuition](#intuition)
2. [Bayes Classifier Foundation](#bayes-foundation)
3. [LDA: Assumptions & Decision Rule](#lda-decision)
4. [LDA: Fisher's Criterion & Scatter Matrices](#lda-fisher)
5. [QDA: Relaxing Equal Covariance](#qda)
6. [LDA as Dimensionality Reduction](#lda-dimred)
7. [Regularized LDA](#rlda)
8. [Connections: LDA / QDA / Naive Bayes / Logistic Regression](#connections)
9. [Setup & Imports](#imports)
10. [Scatter Matrices: S_W and S_B](#scatter-matrices)
11. [LDA Classifier from Scratch](#lda-impl)
12. [QDA Classifier from Scratch](#qda-impl)
13. [LDA as Dimensionality Reduction](#lda-dimred-impl)
14. [Regularized LDA from Scratch](#rlda-impl)
15. [Complete Unified Class](#unified-class)
16. [Demo: Classification (Iris)](#demo-iris)
17. [Decision Boundary Visualization](#decision-boundary)
18. [Demo: LDA Projection (2D → 1D and 3D → 2D)](#demo-projection)
19. [LDA vs QDA: When to Use Which](#lda-vs-qda)
20. [Comparison: LDA / QDA / NB / Logistic Regression](#comparison)
21. [Complexity Analysis](#complexity)
22. [Advantages and Disadvantages](#pros-cons)
23. [Summary](#summary)
24. [Sources](#sources)

<a id="intuition"></a>
## 1. Intuition

Discriminant Analysis classifies by asking: **which class distribution most likely generated this point?**

It is a **generative model**: we fit a multivariate Gaussian to each class, then use Bayes' theorem to assign new points.

**Real-world analogy — face recognition with height and weight:**

You have measurements (height, weight) for male and female athletes. Each group forms a roughly elliptical cloud in 2D. To classify a new measurement, you check which ellipse it falls closest to — adjusting for the fact that one group might be more common (prior).

**LDA** assumes both classes share the **same covariance** matrix → the ellipses are the same shape and orientation → the boundary between them is a **straight line** (hyperplane in higher dimensions).

**QDA** lets each class have its **own covariance** → ellipses have different shapes → the boundary is a **curve** (quadratic surface).

```
LDA:  ellipses same shape  →  linear  boundary
QDA:  ellipses free shape  →  quadratic boundary
```

**Historical note:** LDA was introduced by Ronald Fisher in 1936 in the same paper that introduced the Iris dataset. He framed it as a **projection** problem — find the axis along which the classes are maximally separated — rather than a probabilistic one. Both formulations give the same result.

<a id="bayes-foundation"></a>
## 2. Bayes Classifier Foundation

Discriminant Analysis is a **Bayes classifier** with a specific parametric form for the class-conditional densities.

The Bayes optimal classifier assigns class $c^*$ by:

$$c^* = \underset{c}{\arg\max}\; P(C = c \mid \mathbf{x}) = \underset{c}{\arg\max}\; \underbrace{p(\mathbf{x} \mid C = c)}_{\text{likelihood}} \cdot \underbrace{P(C = c)}_{\text{prior}}$$

Both LDA and QDA model $p(\mathbf{x} \mid C = c)$ as a **multivariate Gaussian**:

$$p(\mathbf{x} \mid C = c) = \frac{1}{(2\pi)^{p/2} |\Sigma_c|^{1/2}} \exp\!\left(-\frac{1}{2}(\mathbf{x} - \boldsymbol{\mu}_c)^T \Sigma_c^{-1} (\mathbf{x} - \boldsymbol{\mu}_c)\right)$$

where $\boldsymbol{\mu}_c \in \mathbb{R}^p$ is the class mean and $\Sigma_c \in \mathbb{R}^{p \times p}$ is the class covariance.

**Parameters estimated from training data:**

$$\hat{\boldsymbol{\mu}}_c = \frac{1}{N_c} \sum_{i: y_i = c} \mathbf{x}_i$$

$$\hat{\Sigma}_c = \frac{1}{N_c} \sum_{i: y_i = c} (\mathbf{x}_i - \hat{\boldsymbol{\mu}}_c)(\mathbf{x}_i - \hat{\boldsymbol{\mu}}_c)^T$$

$$\hat{P}(C = c) = \frac{N_c}{N}$$

The **log posterior** (dropping constants shared across classes) gives a **discriminant function** $\delta_c(\mathbf{x})$. The class with the highest $\delta_c(\mathbf{x})$ wins.

<a id="lda-decision"></a>
## 3. LDA: Assumptions & Decision Rule

### Assumption: Shared Covariance

LDA assumes all $K$ classes share **one** covariance matrix:

$$\Sigma_1 = \Sigma_2 = \cdots = \Sigma_K = \Sigma$$

Estimated as the **pooled within-class covariance**:

$$\hat{\Sigma} = \frac{1}{N - K} \sum_{c=1}^{K} \sum_{i: y_i = c} (\mathbf{x}_i - \hat{\boldsymbol{\mu}}_c)(\mathbf{x}_i - \hat{\boldsymbol{\mu}}_c)^T$$

### Linear Discriminant Function

Taking the log posterior and cancelling the $(2\pi)^{p/2}|\Sigma|^{1/2}$ term (same for all classes under shared $\Sigma$):

$$\delta_c(\mathbf{x}) = \mathbf{x}^T \Sigma^{-1} \hat{\boldsymbol{\mu}}_c - \frac{1}{2} \hat{\boldsymbol{\mu}}_c^T \Sigma^{-1} \hat{\boldsymbol{\mu}}_c + \log \hat{P}(c)$$

This is **linear in $\mathbf{x}$** → linear decision boundary.

**Classification rule:**

$$\hat{c} = \underset{c}{\arg\max}\; \delta_c(\mathbf{x})$$

### Binary LDA Decision Boundary

For two classes $c_1, c_2$, the boundary is where $\delta_{c_1}(\mathbf{x}) = \delta_{c_2}(\mathbf{x})$:

$$\mathbf{x}^T \Sigma^{-1}(\boldsymbol{\mu}_1 - \boldsymbol{\mu}_2) = \frac{1}{2}(\boldsymbol{\mu}_1^T \Sigma^{-1} \boldsymbol{\mu}_1 - \boldsymbol{\mu}_2^T \Sigma^{-1} \boldsymbol{\mu}_2) - \log\frac{P(c_1)}{P(c_2)}$$

Left side is linear in $\mathbf{x}$ → hyperplane.

<a id="lda-fisher"></a>
## 4. LDA: Fisher's Criterion & Scatter Matrices

Fisher (1936) derived LDA from a completely different angle: **find the projection direction that maximally separates the classes**.

### Scatter Matrices

**Within-class scatter** $S_W$ — total spread of points around their class means:

$$S_W = \sum_{c=1}^{K} \sum_{i: y_i = c} (\mathbf{x}_i - \boldsymbol{\mu}_c)(\mathbf{x}_i - \boldsymbol{\mu}_c)^T$$

**Between-class scatter** $S_B$ — spread of class means around the global mean $\boldsymbol{\mu}$:

$$S_B = \sum_{c=1}^{K} N_c (\boldsymbol{\mu}_c - \boldsymbol{\mu})(\boldsymbol{\mu}_c - \boldsymbol{\mu})^T$$

Note: $S_W \propto \hat{\Sigma}_{\text{pooled}}$; they are the same up to a scalar.

### Fisher's Criterion

Find the projection matrix $W \in \mathbb{R}^{p \times d}$ that **maximizes between-class variance relative to within-class variance** in the projected space:

$$W^* = \underset{W}{\arg\max}\; \mathcal{J}(W) = \frac{|W^T S_B W|}{|W^T S_W W|}$$

### Solution: Generalized Eigenvalue Problem

The optimal directions are the eigenvectors of $S_W^{-1} S_B$:

$$S_B \mathbf{w} = \lambda S_W \mathbf{w} \quad \Longleftrightarrow \quad S_W^{-1} S_B \mathbf{w} = \lambda \mathbf{w}$$

Sort eigenvectors by descending eigenvalue → first $d$ eigenvectors form the optimal projection.

**Maximum number of useful discriminant directions:** $\min(K-1, p)$  
- $K-1$: rank of $S_B$ is at most $K-1$ (only $K-1$ independent vectors between $K$ means)
- $p$: can't project into more dimensions than the original space

**Key insight:** for $K=2$ classes, Fisher's solution and the Bayes LDA classifier give the **same** classification result. Fisher's framing generalizes more naturally to dimensionality reduction.

<a id="qda"></a>
## 5. QDA: Relaxing Equal Covariance

QDA removes the shared-covariance assumption. Each class has its **own** covariance $\Sigma_c$.

### Quadratic Discriminant Function

The log posterior no longer cancels the $|\Sigma_c|$ and quadratic terms, giving:

$$\delta_c(\mathbf{x}) = -\frac{1}{2} \log |\Sigma_c| - \frac{1}{2}(\mathbf{x} - \boldsymbol{\mu}_c)^T \Sigma_c^{-1}(\mathbf{x} - \boldsymbol{\mu}_c) + \log P(c)$$

The term $(\mathbf{x} - \boldsymbol{\mu}_c)^T \Sigma_c^{-1}(\mathbf{x} - \boldsymbol{\mu}_c)$ is the **Mahalanobis distance** squared from $\mathbf{x}$ to class $c$.

This function is **quadratic in $\mathbf{x}$** → the decision boundary between two classes is a **quadratic surface** (ellipse, parabola, or hyperbola in 2D).

### LDA vs QDA trade-off

| | LDA | QDA |
|---|---|---|
| Covariances | Shared: $\Sigma$ | Per-class: $\Sigma_c$ |
| Parameters | $p^2/2 + Kp$ | $Kp^2/2 + Kp$ |
| Decision boundary | Linear (hyperplane) | Quadratic (conic section) |
| Bias | Higher (assumes equal $\Sigma$) | Lower |
| Variance | Lower | Higher |
| Works best when | $n$ small, assumption holds | $n$ large per class |
| Can project | Yes (Fisher's criterion) | No |

<a id="lda-dimred"></a>
## 6. LDA as Dimensionality Reduction

LDA is also a **supervised** dimensionality reduction technique. Unlike PCA (which maximizes total variance), LDA maximizes **class separability** in the projected space.

**Algorithm:**
1. Compute $S_W$ and $S_B$
2. Solve $S_W^{-1} S_B W = W \Lambda$
3. Take the top $d \leq K-1$ eigenvectors
4. Project: $Z = X W$

**PCA vs LDA:**

| | PCA | LDA |
|---|---|---|
| Criterion | Max variance (unsupervised) | Max class separation (supervised) |
| Uses labels | No | Yes |
| Max components | $p$ | $K - 1$ |
| Best for | Compression, noise removal | Visualization, preprocessing for classifier |

For a 4-class problem (like Iris), LDA can produce **at most 3 discriminant components**. For 2 classes, exactly **1 discriminant direction** exists.

<a id="rlda"></a>
## 7. Regularized LDA

### The Singularity Problem

$S_W$ (and individual $\Sigma_c$ in QDA) is **singular** when $n < p$ (more features than samples). This makes $S_W^{-1}$ undefined.

### Ridge Shrinkage (Friedman 1989 — RDA)

Regularize by shrinking the sample covariance toward a scaled identity:

$$\hat{\Sigma}_{\text{reg}} = (1 - \gamma)\hat{\Sigma} + \gamma \frac{\text{tr}(\hat{\Sigma})}{p} I$$

- $\gamma = 0$: standard MLE covariance
- $\gamma = 1$: spherical covariance (isotropic Gaussian — equivalent to nearest centroid)
- $\gamma \in (0,1)$: interpolation

This is called **Regularized Discriminant Analysis** (RDA, Friedman 1989). The optimal $\gamma$ is found by cross-validation or by an analytic estimator.

### Analytic Shrinkage Estimators

**Ledoit-Wolf (2004)** derives the asymptotically optimal $\gamma$ by minimizing the Frobenius-norm loss between $\hat{\Sigma}_{\text{reg}}$ and the true $\Sigma$. sklearn implements this as `shrinkage='auto'` with `solver='eigen'`.

**Oracle Approximating Shrinkage (OAS, Chen et al. 2010)** provides a simpler closed-form approximation that outperforms Ledoit-Wolf in finite samples for Gaussian data:

$$\gamma^* = \frac{(1-2/p)\,\text{tr}(\hat{\Sigma}^2) + (\text{tr}\,\hat{\Sigma})^2}{(n+1-2/p)\left[\text{tr}(\hat{\Sigma}^2) - (\text{tr}\,\hat{\Sigma})^2/p\right]}$$

sklearn implements OAS as `solver='lsqr', shrinkage='auto'`.

<a id="connections"></a>
## 8. Connections: LDA / QDA / Naive Bayes / Logistic Regression

These four classifiers are deeply related:

| Classifier | Gaussian? | Equal $\Sigma$? | Independent features? | Boundary |
|---|---|---|---|---|
| QDA | Yes | No | No | Quadratic |
| LDA | Yes | Yes | No | Linear |
| Gaussian NB | Yes | No (diagonal per class) | Yes | Quadratic |
| Logistic Reg | No | — | — | Linear |

**Gaussian NB = diagonal QDA:** Gaussian NB estimates a separate diagonal covariance per class (independence assumption), which is equivalent to QDA restricted to diagonal covariance matrices. If instead all classes share the same diagonal covariance, the result is diagonal LDA — a distinct (and stronger) assumption.

**LDA vs Logistic Regression:** both produce linear boundaries, but:
- LDA estimates $P(\mathbf{x}|C)$ then uses Bayes → more assumptions, more efficient when Gaussian holds
- Logistic regression directly models $P(C|\mathbf{x})$ → fewer assumptions, robust to non-Gaussian features

> Efron (1975) showed that LDA can be substantially more efficient than logistic regression when the Gaussian assumption holds exactly. The efficiency gain depends on the problem dimensionality and sample size; Efron reports that LDA can achieve up to 2× the asymptotic efficiency of logistic regression in favorable conditions.

<a id="imports"></a>
## 9. Setup & Imports

All implementations use NumPy only. sklearn is used for comparison and dataset loading.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.datasets import load_iris, make_classification, make_blobs
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
print("Imports OK")

<a id="scatter-matrices"></a>
## 10. Scatter Matrices: $S_W$ and $S_B$

The scatter matrices are the core building blocks of LDA. We implement them separately before the full classifier so their structure is explicit.

`compute_scatter_matrices` returns $S_W$ (within-class scatter), $S_B$ (between-class scatter), per-class means, and the global mean — all as NumPy arrays. We verify on a toy 3-class 2D dataset.

In [ ]:
def compute_scatter_matrices(X, y):
    """
    Compute within-class (S_W) and between-class (S_B) scatter matrices.

    Returns
    -------
    S_W     : (p, p) within-class scatter
    S_B     : (p, p) between-class scatter
    means   : dict  {class: mean vector}
    mu      : (p,)  global mean
    """
    classes  = np.unique(y)
    p        = X.shape[1]
    mu       = X.mean(axis=0)
    S_W      = np.zeros((p, p))
    S_B      = np.zeros((p, p))
    means    = {}

    for c in classes:
        X_c       = X[y == c]
        mu_c      = X_c.mean(axis=0)
        means[c]  = mu_c

        # Within-class scatter: sum of outer products
        diff = X_c - mu_c                  # (N_c, p)
        S_W += diff.T @ diff               # (p, p)

        # Between-class scatter: N_c * outer product of class-mean diff
        d_mu = (mu_c - mu).reshape(-1, 1)  # (p, 1)
        S_B += len(X_c) * (d_mu @ d_mu.T) # (p, p)

    return S_W, S_B, means, mu


# Verify on simple 3-class dataset
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
S_W, S_B, class_means, global_mean = compute_scatter_matrices(X_iris, y_iris)

print(f"S_W shape: {S_W.shape}")
print(f"S_B shape: {S_B.shape}")
print(f"S_W rank:  {np.linalg.matrix_rank(S_W)} (full rank = {X_iris.shape[1]})")
print(f"S_B rank:  {np.linalg.matrix_rank(S_B)} (max useful = K-1 = {len(np.unique(y_iris))-1})")

We visualize $S_W$ and $S_B$ as heatmaps and print the eigenvalues of $S_W^{-1} S_B$ to confirm there are at most $K-1 = 2$ non-zero eigenvalues.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, mat, title in zip(axes, [S_W, S_B], ['$S_W$ (within-class scatter)', '$S_B$ (between-class scatter)']):
    im = ax.imshow(mat, cmap='RdBu_r', aspect='auto')
    plt.colorbar(im, ax=ax)
    ax.set_title(title)
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(iris.feature_names, rotation=30, ha='right', fontsize=7)
    ax.set_yticklabels(iris.feature_names, fontsize=7)
plt.tight_layout()
plt.show()

# Eigenvalues of S_W^{-1} S_B
eig_vals = np.linalg.eigvals(np.linalg.inv(S_W) @ S_B)
eig_vals = np.sort(np.real(eig_vals))[::-1]
print("Eigenvalues of S_W⁻¹ S_B:", np.round(eig_vals, 4))
print("→ Only 2 non-zero (K-1=2 discriminant directions for 3 classes)")

<a id="lda-impl"></a>
## 11. LDA Classifier from Scratch

We implement LDA as a Bayes classifier using the linear discriminant function $\delta_c(\mathbf{x})$.

`LDAClassifier.fit` computes class means, pooled covariance $\hat{\Sigma}$, and pre-computes $\Sigma^{-1}\mu_c$ and the scalar term $\frac{1}{2}\mu_c^T \Sigma^{-1} \mu_c$ for each class. `predict` evaluates $\delta_c(\mathbf{x})$ for all classes via a single matrix operation.

In [ ]:
class LDAClassifier:
    """
    Linear Discriminant Analysis classifier.

    Decision rule: argmax_c  x^T Σ⁻¹ μ_c  - ½ μ_c^T Σ⁻¹ μ_c  + log P(c)
    """

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        self.classes_  = np.unique(y)
        n, p           = X.shape
        K              = len(self.classes_)

        # Class priors
        self.log_priors_ = np.array([
            np.log(np.sum(y == c) / n) for c in self.classes_
        ])

        # Class means
        self.means_ = np.array([X[y == c].mean(axis=0) for c in self.classes_])

        # Pooled within-class covariance (unbiased: divide by N-K)
        S_W = np.zeros((p, p))
        for c, mu_c in zip(self.classes_, self.means_):
            diff = X[y == c] - mu_c
            S_W += diff.T @ diff
        self.cov_ = S_W / (n - K)

        # Pre-compute Σ⁻¹ μ_c and scalar terms for fast prediction
        Sigma_inv        = np.linalg.inv(self.cov_)
        self.Sigma_inv_  = Sigma_inv
        # coef_[k] = Σ⁻¹ μ_k  →  shape (K, p)
        self.coef_       = (Sigma_inv @ self.means_.T).T
        # intercept_[k] = -½ μ_k^T Σ⁻¹ μ_k + log P(c_k)
        self.intercept_  = (
            -0.5 * np.sum(self.means_ * self.coef_, axis=1)
            + self.log_priors_
        )
        return self

    def decision_function(self, X):
        """δ_c(x) for each class. Shape (n_samples, n_classes)."""
        return X @ self.coef_.T + self.intercept_

    def predict_proba(self, X):
        scores = self.decision_function(X)
        scores -= scores.max(axis=1, keepdims=True)  # numerical stability
        exp_scores = np.exp(scores)
        return exp_scores / exp_scores.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes_[np.argmax(self.decision_function(X), axis=1)]


print("LDAClassifier defined.")

<a id="qda-impl"></a>
## 12. QDA Classifier from Scratch

QDA stores a separate $\Sigma_c$ per class. The discriminant function now includes the $-\frac{1}{2}\log|\Sigma_c|$ term and the full Mahalanobis distance.

`QDAClassifier.fit` stores per-class $\mu_c$, $\Sigma_c$, $\Sigma_c^{-1}$, and $\log|\Sigma_c|$. `predict` evaluates $\delta_c(\mathbf{x})$ for each class using the full quadratic form — no matrix-level shortcut since $\Sigma_c$ varies.

In [ ]:
class QDAClassifier:
    """
    Quadratic Discriminant Analysis classifier.

    Decision rule:
        argmax_c  -½ log|Σ_c|  -  ½ (x-μ_c)^T Σ_c⁻¹ (x-μ_c)  + log P(c)
    """

    def __init__(self, reg_param=0.0):
        """
        reg_param : float — regularization added to each Σ_c diagonal
                    (prevents singularity in small-n settings)
        """
        self.reg_param = reg_param

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        self.classes_    = np.unique(y)
        n                = len(y)
        self.means_      = {}
        self.cov_inv_    = {}
        self.log_det_    = {}
        self.log_priors_ = {}

        for c in self.classes_:
            X_c    = X[y == c]
            mu_c   = X_c.mean(axis=0)
            diff   = X_c - mu_c
            Sigma  = (diff.T @ diff) / len(X_c)
            Sigma += self.reg_param * np.eye(X.shape[1])  # regularization

            self.means_[c]      = mu_c
            self.cov_inv_[c]    = np.linalg.inv(Sigma)
            sign, logdet        = np.linalg.slogdet(Sigma)
            self.log_det_[c]    = logdet
            self.log_priors_[c] = np.log(len(X_c) / n)

        return self

    def decision_function(self, X):
        X = np.asarray(X, dtype=float)
        scores = np.zeros((len(X), len(self.classes_)))
        for k, c in enumerate(self.classes_):
            diff = X - self.means_[c]                        # (n, p)
            mahal_sq = np.einsum('ni,ij,nj->n', diff, self.cov_inv_[c], diff)
            scores[:, k] = -0.5 * self.log_det_[c] - 0.5 * mahal_sq + self.log_priors_[c]
        return scores

    def predict_proba(self, X):
        scores = self.decision_function(X)
        scores -= scores.max(axis=1, keepdims=True)
        exp_s = np.exp(scores)
        return exp_s / exp_s.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes_[np.argmax(self.decision_function(X), axis=1)]


print("QDAClassifier defined.")

<a id="lda-dimred-impl"></a>
## 13. LDA as Dimensionality Reduction

Fisher's formulation of LDA computes the top eigenvectors of $S_W^{-1} S_B$ to find axes of maximum class separation.

`LDATransformer.fit` solves the generalized eigenvalue problem $S_W^{-1} S_B W = W \Lambda$ and keeps the top $d$ eigenvectors sorted by descending eigenvalue. `transform` projects $X$ onto the discriminant subspace.

In [ ]:
class LDATransformer:
    """
    LDA for supervised dimensionality reduction.

    Projects data onto the top d Fisher discriminant directions.

    Parameters
    ----------
    n_components : int or None — number of components to keep (max K-1)
    """

    def __init__(self, n_components=None):
        self.n_components = n_components

    def fit(self, X, y):
        from scipy.linalg import eigh
        X = np.asarray(X, dtype=float)
        K = len(np.unique(y))

        S_W, S_B, _, _ = compute_scatter_matrices(X, y)

        # Solve generalized symmetric eigenvalue problem: S_B w = λ S_W w
        # eigh guarantees real eigenvalues; returns ascending order
        eig_vals, eig_vecs = eigh(S_B, S_W)

        # Sort by descending eigenvalue
        order    = np.argsort(eig_vals)[::-1]
        eig_vals = eig_vals[order]
        eig_vecs = eig_vecs[:, order]

        # Number of useful components is at most K-1
        max_components  = K - 1
        n_comp          = self.n_components or max_components
        n_comp          = min(n_comp, max_components)

        self.components_     = eig_vecs[:, :n_comp].T   # (n_comp, p)
        self.explained_var_  = eig_vals[:n_comp]
        self.explained_ratio_ = eig_vals[:n_comp] / (eig_vals.sum() + 1e-15)
        return self

    def transform(self, X):
        """Project X onto discriminant axes. Returns shape (n_samples, n_components)."""
        return np.asarray(X, dtype=float) @ self.components_.T

    def fit_transform(self, X, y):
        return self.fit(X, y).transform(X)


print("LDATransformer defined.")

<a id="rlda-impl"></a>
## 14. Regularized LDA from Scratch

We extend `LDAClassifier` with a shrinkage parameter $\gamma$ that blends the sample covariance toward a scaled identity.

`RLDAClassifier` applies $\hat{\Sigma}_{\text{reg}} = (1-\gamma)\hat{\Sigma} + \gamma \cdot \frac{\text{tr}(\hat{\Sigma})}{p} I$ before inverting. All other steps are identical to LDA.

In [ ]:
class RLDAClassifier(LDAClassifier):
    """
    Regularized LDA with shrinkage toward scaled identity.

    Sigma_reg = (1-gamma) * Sigma + gamma * (tr(Sigma)/p) * I

    Parameters
    ----------
    gamma : float in [0, 1]  (0 = standard LDA, 1 = nearest centroid)
    """

    def __init__(self, gamma=0.0):
        self.gamma = gamma

    def fit(self, X, y):
        # Compute pooled covariance as usual
        super().fit(X, y)
        p     = self.cov_.shape[0]
        Sigma = self.cov_

        # Shrink
        target       = (np.trace(Sigma) / p) * np.eye(p)
        Sigma_reg    = (1 - self.gamma) * Sigma + self.gamma * target
        self.cov_    = Sigma_reg

        # Re-compute coefficients with regularized covariance
        Sigma_inv        = np.linalg.inv(Sigma_reg)
        self.Sigma_inv_  = Sigma_inv
        self.coef_       = (Sigma_inv @ self.means_.T).T
        self.intercept_  = (
            -0.5 * np.sum(self.means_ * self.coef_, axis=1)
            + self.log_priors_
        )
        return self


print("RLDAClassifier defined.")

<a id="unified-class"></a>
## 15. Complete Unified Class

A single sklearn-style `DiscriminantAnalysis` class dispatches to the appropriate implementation.

`DiscriminantAnalysis` is a thin dispatcher: it selects `LDAClassifier`, `RLDAClassifier`, or `QDAClassifier` at `__init__` time based on the `kind` parameter.

In [ ]:
class DiscriminantAnalysis:
    """
    Unified interface for LDA, Regularized LDA, and QDA.

    Parameters
    ----------
    kind         : 'lda', 'rlda', or 'qda'
    gamma        : shrinkage for RLDA  (ignored otherwise)
    reg_param    : diagonal regularization for QDA  (ignored otherwise)
    """

    def __init__(self, kind='lda', gamma=0.0, reg_param=0.0):
        if kind == 'lda':
            self._clf = LDAClassifier()
        elif kind == 'rlda':
            self._clf = RLDAClassifier(gamma=gamma)
        elif kind == 'qda':
            self._clf = QDAClassifier(reg_param=reg_param)
        else:
            raise ValueError("kind must be 'lda', 'rlda', or 'qda'")
        self.kind = kind

    def fit(self, X, y):              return self._clf.fit(X, y)
    def predict(self, X):             return self._clf.predict(X)
    def predict_proba(self, X):       return self._clf.predict_proba(X)
    def decision_function(self, X):   return self._clf.decision_function(X)


print("DiscriminantAnalysis wrapper defined.")

<a id="demo-iris"></a>
## 16. Demo: Classification (Iris)

We evaluate LDA, RLDA, and QDA on Iris (150 samples, 4 features, 3 classes) and compare accuracy, confusion matrices, and per-class posteriors against sklearn.

Load Iris, 80/20 split. Train LDA, QDA, and RLDA (γ=0.5). Compare against sklearn's implementations.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42)

results = {}

for name, clf in [
    ('LDA (ours)',      DiscriminantAnalysis(kind='lda')),
    ('RLDA γ=0.5 (ours)', DiscriminantAnalysis(kind='rlda', gamma=0.5)),
    ('QDA (ours)',      DiscriminantAnalysis(kind='qda')),
    ('LDA (sklearn)',   LinearDiscriminantAnalysis()),
    ('QDA (sklearn)',   QuadraticDiscriminantAnalysis()),
]:
    clf.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, clf.predict(X_te))
    results[name] = acc
    print(f"{name:22s}  accuracy = {acc:.4f}")

We plot confusion matrices for LDA and QDA side by side to compare which class pairs each model struggles with.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (name, kind) in zip(axes, [('LDA', 'lda'), ('QDA', 'qda')]):
    clf = DiscriminantAnalysis(kind=kind)
    clf.fit(X_tr, y_tr)
    cm = confusion_matrix(y_te, clf.predict(X_te))
    disp = ConfusionMatrixDisplay(cm, display_labels=iris.target_names)
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(f'{name} — Iris')

plt.tight_layout()
plt.show()

<a id="decision-boundary"></a>
## 17. Decision Boundary Visualization

Using 2 features shows the qualitative difference between LDA's straight boundary and QDA's curved one. We use a synthetic dataset where the covariances differ to make the effect clear.

`plot_da_boundary` plots the dense meshgrid prediction as a colored background and overlays training points. We call it with LDA, QDA, and optionally Gaussian NB on the same data to see the different boundary shapes.

In [ ]:
def plot_da_boundary(clf, X, y, title='', h=0.05):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                          np.arange(y_min, y_max, h))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    n_cls = len(np.unique(y))
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.contourf(xx, yy, Z, alpha=0.35, cmap=plt.cm.Pastel1,
                levels=np.arange(-0.5, n_cls + 0.5))
    ax.contour(xx, yy, Z, colors='k', linewidths=0.8, alpha=0.4)

    cmap = plt.cm.Set1
    for c in np.unique(y):
        m = y == c
        ax.scatter(X[m, 0], X[m, 1], c=[cmap(c / n_cls)],
                   edgecolors='k', linewidth=0.5, s=50, label=f'Class {c}')
    ax.set_title(title); ax.legend()
    plt.tight_layout(); plt.show()


# Synthetic: two classes with very different covariances → boundary should be curved
np.random.seed(1)
X0 = np.random.multivariate_normal([0, 0], [[1.5, 0.8], [0.8, 0.5]], 150)
X1 = np.random.multivariate_normal([3, 3], [[0.4, -0.3], [-0.3, 1.5]], 150)
X_syn = np.vstack([X0, X1])
y_syn = np.array([0]*150 + [1]*150)

for name, clf in [
    ('LDA (linear boundary)',    DiscriminantAnalysis(kind='lda')),
    ('QDA (quadratic boundary)', DiscriminantAnalysis(kind='qda')),
]:
    clf.fit(X_syn, y_syn)
    plot_da_boundary(clf, X_syn, y_syn, title=name)

We also run both models on the 2-feature Iris data (petal features) to compare their boundaries on a familiar dataset.

In [ ]:
X_2d, y_2d = X_iris[:, 2:], y_iris  # petal length, petal width
for name, kind in [('LDA — Iris petals', 'lda'), ('QDA — Iris petals', 'qda')]:
    clf = DiscriminantAnalysis(kind=kind)
    clf.fit(X_2d, y_2d)
    plot_da_boundary(clf, X_2d, y_2d, title=name)

<a id="demo-projection"></a>
## 18. Demo: LDA Projection

LDA's most powerful feature as a dimensionality reduction tool is that it concentrates class-discriminative information into just $K-1$ dimensions.

**Iris has 3 classes → at most 2 discriminant components.** We project from 4D to 2D using `LDATransformer` and compare with PCA, showing that LDA produces better class separation visually.

In [ ]:
from sklearn.decomposition import PCA

# LDA projection: 4D → 2D
lda_tr = LDATransformer(n_components=2)
X_lda  = lda_tr.fit_transform(X_iris, y_iris)

# PCA projection: 4D → 2D (for comparison)
pca    = PCA(n_components=2)
X_pca  = pca.fit_transform(X_iris)

print(f"LDA explained variance ratio: {lda_tr.explained_ratio_}")
print(f"PCA explained variance ratio: {pca.explained_variance_ratio_}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for ax, X_proj, title, xl, yl in [
    (axes[0], X_lda, 'LDA Projection (supervised)', 'LD1', 'LD2'),
    (axes[1], X_pca, 'PCA Projection (unsupervised)', 'PC1', 'PC2'),
]:
    for cls, col, name in zip([0,1,2], colors, iris.target_names):
        mask = y_iris == cls
        ax.scatter(X_proj[mask, 0], X_proj[mask, 1],
                   c=col, label=name, edgecolors='k', linewidth=0.4, s=50)
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    ax.set_title(title); ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('LDA (supervised) vs PCA (unsupervised) on Iris', fontsize=13)
plt.tight_layout()
plt.show()

For a binary classification problem, LDA has exactly 1 discriminant direction. We project a 4-feature dataset to 1D and visualize the class distributions as overlapping histograms.

In [ ]:
# Binary subset: Iris class 0 vs class 1
mask_01 = y_iris < 2
X_bin   = X_iris[mask_01]
y_bin   = y_iris[mask_01]

lda_1d   = LDATransformer(n_components=1)
X_proj1d = lda_1d.fit_transform(X_bin, y_bin).ravel()

plt.figure(figsize=(8, 4))
for cls, col, name in zip([0, 1], ['steelblue', 'tomato'], ['Setosa', 'Versicolor']):
    plt.hist(X_proj1d[y_bin == cls], bins=20, alpha=0.6, color=col, label=name)
plt.xlabel('LD1 projection value')
plt.ylabel('Count')
plt.title('LDA 1D Projection — Iris Setosa vs Versicolor')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<a id="lda-vs-qda"></a>
## 19. LDA vs QDA: When to Use Which

The key trade-off is **bias vs variance** driven by the shared-covariance assumption.

We generate two synthetic experiments:
1. **Equal covariances** — LDA assumption holds → LDA should win or tie
2. **Unequal covariances** — QDA assumption holds → QDA should win

We vary the training set size to show the bias-variance crossover point.

In [ ]:
def make_binary_gaussian(n, equal_cov, seed=0):
    rng = np.random.RandomState(seed)
    cov0 = [[1.5, 0.5], [0.5, 1.0]]
    cov1 = cov0 if equal_cov else [[0.3, -0.2], [-0.2, 1.8]]
    X0   = rng.multivariate_normal([0, 0], cov0, n)
    X1   = rng.multivariate_normal([2, 2], cov1, n)
    return np.vstack([X0, X1]), np.array([0]*n + [1]*n)


ns      = [20, 50, 100, 250, 500, 1000]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, equal, title in zip(axes,
                              [True, False],
                              ['Equal covariances (LDA assumption holds)',
                               'Unequal covariances (QDA assumption holds)']):
    lda_accs, qda_accs = [], []
    for n in ns:
        X_data, y_data = make_binary_gaussian(n, equal)
        Xtr, Xte, ytr, yte = train_test_split(X_data, y_data, test_size=0.3, random_state=1)
        for accs, kind in [(lda_accs, 'lda'), (qda_accs, 'qda')]:
            clf = DiscriminantAnalysis(kind=kind, reg_param=1e-6)
            clf.fit(Xtr, ytr)
            accs.append(accuracy_score(yte, clf.predict(Xte)))

    ax.plot(ns, lda_accs, 'b-o', label='LDA')
    ax.plot(ns, qda_accs, 'r-s', label='QDA')
    ax.set_xlabel('Training samples per class')
    ax.set_ylabel('Test accuracy')
    ax.set_title(title)
    ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('LDA vs QDA: accuracy vs training size', fontsize=12)
plt.tight_layout()
plt.show()

<a id="comparison"></a>
## 20. Comparison: LDA / QDA / Gaussian NB / Logistic Regression

We compare all four classifiers on the same benchmark to see when each excels.

We run 5-fold cross-validation for all four classifiers on three datasets: Iris (multi-class, small), a high-correlation synthetic set, and a non-Gaussian synthetic set. We display a bar chart of CV accuracy.

In [ ]:
from sklearn.pipeline import make_pipeline

# We use sklearn versions for CV compatibility
classifiers = {
    'LDA':           LinearDiscriminantAnalysis(),
    'QDA':           QuadraticDiscriminantAnalysis(reg_param=1e-6),
    'Gaussian NB':   GaussianNB(),
    'Logistic Reg':  make_pipeline(StandardScaler(), LogisticRegression(max_iter=500)),
}

# Three datasets
np.random.seed(7)
X_corr, y_corr = make_classification(n_samples=300, n_features=4,
                                      n_informative=2, n_redundant=2, random_state=5)
X_nongauss = np.vstack([
    np.random.exponential(1.0, (200, 4)),
    np.random.exponential(2.0, (200, 4)) + 1.5
])
y_nongauss = np.array([0]*200 + [1]*200)

datasets = [
    (X_iris,    y_iris,    'Iris (Gaussian)'),
    (X_corr,    y_corr,    'Correlated features'),
    (X_nongauss, y_nongauss, 'Non-Gaussian (Exponential)'),
]

x_pos = np.arange(len(classifiers))
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for ax, (X_d, y_d, dname) in zip(axes, datasets):
    scores = []
    for clf in classifiers.values():
        cv = cross_val_score(clf, X_d, y_d, cv=5, scoring='accuracy')
        scores.append(cv.mean())
    bars = ax.bar(x_pos, scores, color=['steelblue','tomato','seagreen','darkorange'])
    for bar, sc in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{sc:.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(classifiers.keys(), rotation=20, ha='right', fontsize=8)
    ax.set_title(dname)
    ax.set_ylim(0.4, 1.05)
    ax.set_ylabel('5-fold CV accuracy')
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('LDA / QDA / Gaussian NB / Logistic Regression Comparison', fontsize=12)
plt.tight_layout()
plt.show()

<a id="complexity"></a>
## 21. Complexity Analysis

| Phase | LDA | QDA | Notes |
|---|---|---|---|
| **Training** | $O(np^2 + p^3)$ | $O(np^2 + Kp^3)$ | $p^2$ for scatter, $p^3$ for inversion |
| **Prediction** | $O(Kp)$ per sample | $O(Kp^2)$ per sample | LDA: dot product; QDA: matrix-vector |
| **Parameters** | $Kp + p^2/2$ | $Kp + Kp^2/2$ | QDA stores $K$ covariance matrices |
| **LDA reduction** | $O(p^3 + Kp^2)$ | — | Eigendecomposition of $S_W^{-1} S_B$ |

**Singular issues:**
- $S_W$ singular when $p > n - K$ → use regularization (`gamma > 0`) or PCA first
- $\Sigma_c$ singular in QDA when $p > N_c$ → use `reg_param > 0`

**Memory:**
- LDA stores one $p \times p$ matrix — $O(p^2)$
- QDA stores $K$ matrices — $O(Kp^2)$: expensive when $K$ and $p$ are large

<a id="pros-cons"></a>
## 22. Advantages and Disadvantages

### LDA

| | Detail |
|---|---|
| **Fast** | Linear prediction $O(Kp)$; no iterations |
| **Dimensionality reduction** | Projects to $K-1$ dimensions — best supervised reducer for Gaussian data |
| **Works with small data** | Shares covariance → fewer parameters |
| **Probabilistic** | Returns calibrated posterior probabilities |
| **Gaussian assumption** | Fails on heavy-tailed, skewed, or multimodal distributions |
| **Equal covariance** | Biased if classes have very different spreads |
| **Singular $S_W$** | Requires regularization when $p \approx n$ |

### QDA

| | Detail |
|---|---|
| **Flexible boundary** | Quadratic → can model elliptical class regions |
| **No shared $\Sigma$ assumption** | Unbiased when classes differ in spread |
| **More parameters** | $Kp^2/2$ → overfits with small $N_c$ |
| **No reduction** | Cannot project to lower dimensions |
| **Singular $\Sigma_c$** | Each class needs enough samples to estimate its covariance |

<a id="summary"></a>
## 23. Summary

```
Both LDA and QDA are generative classifiers:
  model P(x | C=c) = N(x; μ_c, Σ_c)  then apply Bayes

LDA (shared Σ):
  δ_c(x) = x^T Σ⁻¹ μ_c  -  ½ μ_c^T Σ⁻¹ μ_c  +  log P(c)
  → linear in x  →  hyperplane boundary
  Parameters: μ_c, Σ (pooled), P(c)

QDA (per-class Σ_c):
  δ_c(x) = -½ log|Σ_c|  -  ½ (x-μ_c)^T Σ_c⁻¹ (x-μ_c)  +  log P(c)
  → quadratic in x  →  conic section boundary
  Parameters: μ_c, Σ_c, P(c)

Fisher's LDA:
  W* = argmax |W^T S_B W| / |W^T S_W W|
  Solution: eigenvectors of S_W⁻¹ S_B  (top K-1)
  → supervised dimensionality reduction

Regularization:
  Σ_reg = (1-γ)Σ + γ·(tr(Σ)/p)·I
  γ=0: standard LDA; γ=1: nearest centroid

Family tree:
  QDA (per-class Gaussian)
    ↓ shared Σ
  LDA (shared Gaussian)
    ↓ diagonal Σ (independence)
  Gaussian Naive Bayes
```

<a id="sources"></a>
## 24. Sources

### Foundational Papers
- **Fisher, R.A. (1936)**. The Use of Multiple Measurements in Taxonomic Problems. *Annals of Eugenics*, 7(2), 179–188. [DOI](https://doi.org/10.1111/j.1469-1809.1936.tb02137.x) — Original LDA paper and Iris dataset.
- **Rao, C.R. (1948)**. The Utilization of Multiple Measurements in Problems of Biological Classification. *Journal of the Royal Statistical Society B*, 10(2), 159–203. — Extension to multi-class LDA.
- **Friedman, J.H. (1989)**. Regularized Discriminant Analysis. *Journal of the American Statistical Association*, 84(405), 165–175. [DOI](https://doi.org/10.2307/2289860) — Introduced RDA (shrinkage between LDA and QDA).
- **Ledoit, O. & Wolf, M. (2004)**. A well-conditioned estimator for large-dimensional covariance matrices. *Journal of Multivariate Analysis*, 88(2), 365–411. — Optimal analytic shrinkage estimator.
- **Chen, Y., Wiesel, A., Eldar, Y.C. & Hero, A.O. (2010)**. Shrinkage Algorithms for MMSE Covariance Estimation. *IEEE Transactions on Signal Processing*, 58(10), 5016–5029. — Oracle Approximating Shrinkage (OAS) estimator.
- **Efron, B. (1975)**. The Efficiency of Logistic Regression Compared to Normal Discriminant Analysis. *Journal of the American Statistical Association*, 70(352), 892–898. — LDA vs Logistic Regression efficiency.

### Textbooks
- Hastie, Tibshirani, Friedman. ***The Elements of Statistical Learning*** (2nd ed.). Chapter 4: Linear Methods for Classification. [Free PDF](https://web.stanford.edu/~hastie/ElemStatLearn/)
- James, Witten, Hastie, Tibshirani. ***An Introduction to Statistical Learning***. Chapter 4. [Free PDF](https://www.statlearning.com/)
- Bishop, C.M. ***Pattern Recognition and Machine Learning***. Chapter 4.1.
- Murphy, K.P. ***Machine Learning: A Probabilistic Perspective***. Chapter 4.

### Online Resources
- [scikit-learn — Linear and Quadratic Discriminant Analysis](https://scikit-learn.org/stable/modules/lda_qda.html)
- [Wikipedia — Linear discriminant analysis](https://en.wikipedia.org/wiki/Linear_discriminant_analysis)
- [Wikipedia — Quadratic classifier](https://en.wikipedia.org/wiki/Quadratic_classifier)